# Meteorology features

Pull ERA5 meteorology and interpolate it onto both the station
coordinates and the analysis grid centroids from notebook 04. Relative humidity and wind
speed are derived from the raw ERA5 variables rather than pulled directly.

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import cdsapi
import yaml

import monitoring_deserts.data.fetch_era5 as fetch_era5

## Configuration

In [34]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)

RAW_DATA_DIR = Path(gen_config["data_dir"]["raw"]) / "meteorology"
CLEAN_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "meteorology"
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) # The data/processed/ should have no subdirs
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Spacetime settings
with open("../configs/census_scope.yaml") as f:
    census_config = yaml.safe_load(f)

BOUNDING_BOX = tuple(census_config["bounding_box"].values())
DAY_START, DAY_END = census_config["period"]["start"], census_config["period"]["end"]

# Open the analysis df (stations x months, with satellite features) from notebook 3,
# and the analysis grid from notebook 4
analysis_df = pd.read_parquet(CLEAN_DATA_DIR.parent / "satellite" / "analysis_df.parquet")
analysis_grid = gpd.read_parquet(PROCESSED_DATA_DIR / "grid.parquet")

**WARNING:** `BOUNDING_BOX` is `(min_lon, min_lat, max_lon, max_lat)`, this project's usual convention. The `cdsapi` ERA5 request's `area` parameter (Section 5.3) uses a *different* order and convention: `[North, West, South, East]`. Reorder `BOUNDING_BOX` explicitly when building that request -- don't pass it through as-is.

## Step 1 - Pull the data from ERA5

In [36]:
TYPE = "monthly_averaged_reanalysis"
VARIABLES = ["2m_temperature",
             "2m_dewpoint_temperature",
             "10m_u_component_of_wind",
             "10m_v_component_of_wind",
             "boundary_layer_height",
             "total_precipitation",
             "surface_pressure",
            ]
TIME = "00:00"
BBOX = BOUNDING_BOX[3], BOUNDING_BOX[0], BOUNDING_BOX[1], BOUNDING_BOX[2]
D_FORMAT = "netcdf"
REQ_NAME = "reanalysis-era5-single-levels-monthly-means"
groups = fetch_era5.build_year_month_groups(DAY_START, DAY_END)

# Build requests and pull
era5_client = cdsapi.Client()
for year, months in groups:
    fname = RAW_DATA_DIR / f"era5_reanalysis_{year}.netcdf"
    req = fetch_era5.build_request(product_type=TYPE,
                                   variables=VARIABLES,
                                   years=year,
                                   months=months,
                                   time=TIME,
                                   bbox=BBOX,
                                   data_format=D_FORMAT
                                  )
    fetch_era5.pull_era5(era5_client, fname, REQ_NAME, req)

2026-09-26 16:53:49,064 INFO Request ID is 3c3c573f-ffa1-4239-8206-7e3b52ea0325
2026-09-26 16:53:49,200 INFO status has been updated to successful


b710a6c62abc638fa5552ff5c97bf326.zip:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

2026-09-26 16:53:50,449 INFO Request ID is fc2c3529-6a6d-4071-b928-b19c456a63ca
2026-09-26 16:53:50,495 INFO status has been updated to successful


6af36503b7b1c9f4de8062b6e85e2f28.zip:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

## Step 2 - Build meteorology dataset

In [46]:
# Open the pulled datasets
era5_2025 = fetch_era5.open_dataset(RAW_DATA_DIR / "era5_reanalysis_2025.netcdf")
era5_2026 = fetch_era5.open_dataset(RAW_DATA_DIR / "era5_reanalysis_2026.netcdf")

# Combine them
meteo_ds = fetch_era5.concat_datasets(era5_2025, era5_2026)
meteo_ds

<xarray.Dataset> Size: 8MB
Dimensions:     (valid_time: 16, latitude: 132, longitude: 132)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 128B 2025-03-01 ... 2026-06-01
    expver      (valid_time) <U4 256B '0001' '0001' '0001' ... '0001' '0001'
  * latitude    (latitude) float64 1kB 38.25 38.0 37.75 37.5 ... 6.0 5.75 5.5
  * longitude   (longitude) float64 1kB 60.5 60.75 61.0 ... 92.75 93.0 93.25
    number      int64 8B 0
Data variables:
    tp          (valid_time, latitude, longitude) float32 1MB 0.0001869 ... 0...
    t2m         (valid_time, latitude, longitude) float32 1MB 287.6 ... 301.1
    d2m         (valid_time, latitude, longitude) float32 1MB 274.4 ... 297.8
    u10         (valid_time, latitude, longitude) float32 1MB -0.9973 ... 3.221
    v10         (valid_time, latitude, longitude) float32 1MB -0.3357 ... 3.342
    blh         (valid_time, latitude, longitude) float32 1MB 555.0 ... 608.4
    sp          (valid_time, latitude, longitude) float32 1MB 1.002e+05 ... 1...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-09-26T14:41 GRIB to CDM+CF via cfgrib-0.9.1...

## Step 3 - Feature engineering
Calculate relative humidity and wind speed

In [48]:
def relative_humidity(t2m, td2m):
    # ERA5 gives temps in Kelvin, the formula requires Celsius
    t2m_c = t2m - 273
    td2m_c = td2m - 273
    a, b = 17.625, 243.04
    es = np.exp((a * t2m_c) / (b + t2m_c))
    e = np.exp((a * td2m_c) / (b + td2m_c))
    return 100 * e / es

def wind_magnitude(u, v):
    return np.sqrt(u ** 2 + v ** 2)

meteo_ds["relative_humidity"] = relative_humidity(meteo_ds.t2m, meteo_ds.d2m)
meteo_ds["wind_magnitude"] = wind_magnitude(meteo_ds.u10, meteo_ds.v10)

## Step 4 - Interpolate onto station coordinates

In [68]:
# Open analysis_df which has the station coordinates
analysis_df = pd.read_parquet("../data/clean/satellite/analysis_df.parquet")

# Interpolate
unique_stations = analysis_df[["station_id", "longitude", "latitude"]].drop_duplicates("station_id")
station_meteo = fetch_era5.interpolate(meteo_ds, unique_stations, dimname="station")

# Cleanup and assign month coordinate and dimension
station_meteo = fetch_era5.cleanup_dataset(station_meteo)

# Assign station ID as dimension
station_meteo = station_meteo.assign_coords(station_id=("station", unique_stations.station_id)) # assign it as a dimension
station_meteo = station_meteo.swap_dims({"station":"station_id"})

print(station_meteo)

# Save intermediate station meteo dataset
station_meteo.to_netcdf(CLEAN_DATA_DIR / "station_meteo.netcdf")

<xarray.Dataset> Size: 408kB
Dimensions:                (month: 16, station_id: 344)
Coordinates:
  * month                  (month) datetime64[ns] 128B 2025-03-01 ... 2026-06-01
    expver                 (month) <U4 256B '0001' '0001' ... '0001' '0001'
    valid_time             (month) datetime64[ns] 128B 2025-03-01 ... 2026-06-01
  * station_id             (station_id) int64 3kB 50 235 860 ... 3410004 3410005
    longitude              (station_id) float64 3kB 77.13 77.32 ... 76.91 73.85
    latitude               (station_id) float64 3kB 28.67 28.65 ... 28.32 18.46
    station                (station_id) int64 3kB 0 1 2 3 4 ... 340 341 342 343
    number                 int64 8B 0
Data variables:
    total_precipitation    (month, station_id) float64 44kB 0.0003547 ... 0.0...
    temperature            (month, station_id) float64 44kB 296.2 ... 300.4
    dew_point              (month, station_id) float64 44kB 285.9 ... 295.1
    u_wind                 (month, station_id) float64 4

In [69]:
# Merge onto the analysis dataframe
smet_df = (station_meteo
           .to_dataframe()
           .reset_index()
           .drop(columns=["station", "longitude", "latitude", "expver", "number", "valid_time"])
          )
smet_df["month"] = smet_df["month"].dt.to_period("M")
new_analysis_df = analysis_df.merge(smet_df, on=["station_id", "month"])
new_analysis_df.head()

,month,station_id,longitude,latitude,no2,no2_sd,no2_units,aai,aai_sd,aai_units,pm25,pm25_sd,pm25_units,pm25_nan_reason,total_precipitation,temperature,dew_point,u_wind,v_wind,boundary_layer_height,surface_pressure,relative_humidity,wind_magnitude
0,2025-03,50,77.131023,28.674045,0.000106,0.000004,mol/m2,0.208134,0.009537,dimensionless,78.702397,25.853544,ugm3,none,0.000355,296.169999,285.891946,1.188648,-1.215477,502.741195,98572.145820,52.448721,1.701274
1,2025-03,235,77.316032,28.646835,0.000134,0.000005,mol/m2,0.236869,0.021313,dimensionless,81.005496,31.516710,ugm3,none,0.000353,296.211719,285.973674,1.306716,-1.258353,503.958572,98598.230938,52.584996,1.815828
2,2025-03,860,78.005981,27.198658,0.000058,0.000001,mol/m2,0.331405,0.024480,dimensionless,57.440896,17.897299,ugm3,none,0.000115,297.523394,286.185682,0.837137,-0.941065,481.071154,99138.629691,49.242951,1.259937
3,2025-03,2456,80.891736,26.833997,0.000051,0.000001,mol/m2,0.428689,0.022733,dimensionless,58.161202,20.561508,ugm3,none,0.000024,297.821274,285.790882,2.046100,-0.949337,559.689115,99596.172373,47.142067,2.256312
4,2025-03,5408,80.518167,16.515083,0.000067,0.000004,mol/m2,-0.063205,0.019414,dimensionless,24.145945,15.724409,ugm3,none,0.000036,303.195387,294.690451,-0.471233,1.819787,764.035770,100507.773666,60.433367,1.880140


In [71]:
# Write the updated analysis df
new_analysis_df.to_parquet(CLEAN_DATA_DIR / "analysis_df.parquet")

## Step 5 - Interpolate onto the analysis grid